# 45 · AP Invoice Processor

**What is 3-way matching in accounts payable?**

Before an AP team pays a vendor invoice, they verify three documents agree:

1. **Purchase Order (PO)** — what the company authorised to buy and at what price
2. **Goods Receipt (GR)** — what was actually delivered and accepted
3. **Vendor Invoice** — what the vendor is billing

If quantities or prices disagree, payment is held pending review.
The severity and invoice total determine *who* must approve the exception.

**Why combine LLM and deterministic logic?**

- LLMs are good at *extracting* structure from messy free-text invoices
- LLMs are good at *reasoning* about which discrepancy rule fires
- Approval routing rules are precise business policy — hard-code them, never guess

In [ ]:
# Install dependencies (Colab only)
try:
    import google.colab  # noqa: F401
    %pip install -q langchain-openai langchain-core pydantic python-dotenv
except ImportError:
    pass

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"

## Part 1 — The Business Problem

AP teams receive vendor invoices like this:

> *ACME Tech Supplies — Invoice INV-ACME-2025-0892 — PO-2025-001 — 4 cloud servers at $2,100.00 each — Total $8,400.00*

They must check:
- Did we order this? (PO lookup)
- Did we receive it? (GR lookup)
- Does the price match? (price variance check)
- Does the quantity match? (quantity short check)

At scale this is thousands of invoices per month.
Automation frees the team to focus only on genuine exceptions.

## Part 2 — Schema

Three Pydantic models carry all the data through the pipeline.

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class InvoiceLine(BaseModel):
    description: str = Field(description="Line item description")
    quantity: float = Field(description="Quantity of units billed", gt=0)
    unit_price: float = Field(description="Price per unit", gt=0)
    line_total: float = Field(description="quantity * unit_price", gt=0)


class ExtractedInvoice(BaseModel):
    vendor_id: str = Field(description="Vendor identifier e.g. ACME-001")
    invoice_number: str = Field(description="Unique invoice number")
    invoice_date: str = Field(description="Date in YYYY-MM-DD format")
    po_reference: str = Field(description="PO reference e.g. PO-2025-001")
    total_amount: float = Field(description="Total invoice amount", gt=0)
    currency: str = Field(default="USD", description="ISO 4217 currency code")
    lines: List[InvoiceLine] = Field(description="Invoice line items")


class Discrepancy(BaseModel):
    discrepancy_type: Literal[
        "quantity_short", "price_variance", "duplicate", "gr_missing", "po_not_found"
    ] = Field(description="Category of discrepancy")
    field: str = Field(description="Field where discrepancy exists")
    invoice_value: str = Field(description="Value on the invoice")
    expected_value: str = Field(description="Value from PO or GR")
    variance_pct: Optional[float] = Field(default=None, description="% variance")
    severity: Literal["info", "warn", "block"] = Field(description="Severity level")


class MatchResult(BaseModel):
    invoice_number: str
    po_reference: str
    match_status: Literal["clean", "discrepancy", "blocked"]
    discrepancies: List[Discrepancy] = Field(default_factory=list)
    approval_tier: Literal[
        "auto_approve", "line_manager", "finance_controller", "vp_finance"
    ]
    approval_rationale: str


print("Schemas defined.")

## Part 3 — Mock PO and GR Lookup (Deterministic)

In production these would be ERP queries.
Here we use static dicts — the logic is identical; only the data source changes.

In [ ]:
PURCHASE_ORDERS = {
    "PO-2025-001": {"vendor_id": "ACME-001", "quantity": 4, "unit_price": 2100.00, "total_value": 8400.00},
    "PO-2025-002": {"vendor_id": "BETA-002", "quantity": 100, "unit_price": 150.00, "total_value": 15000.00},
    "PO-2025-003": {"vendor_id": "GAMMA-003", "quantity": 75, "unit_price": 375.00, "total_value": 28125.00},
    "PO-2025-004": {"vendor_id": "DELTA-004", "quantity": 50, "unit_price": 440.00, "total_value": 22000.00},
}

GOODS_RECEIPTS = {
    "PO-2025-001": {"qty_received": 4, "receipt_date": "2025-06-10"},
    "PO-2025-002": {"qty_received": 85, "receipt_date": "2025-06-12"},
    # PO-2025-003 intentionally absent: no GR for consulting services
    "PO-2025-004": {"qty_received": 50, "receipt_date": "2025-06-14"},
}


def lookup_po(po_number):
    return PURCHASE_ORDERS.get(po_number)


def lookup_gr(po_number):
    return GOODS_RECEIPTS.get(po_number)


# Quick test
print("PO-2025-001:", lookup_po("PO-2025-001"))
print("GR-PO-2025-002:", lookup_gr("PO-2025-002"))
print("GR-PO-2025-003:", lookup_gr("PO-2025-003"))  # None -- no GR

## Part 4 — The 3-Stage Workflow

**Stage 1**: LLM extracts `ExtractedInvoice` from free text  
**Stage 2**: Deterministic `lookup_po` / `lookup_gr`  
**Stage 3**: LLM compares all three sources and returns `MatchResult`

In [ ]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

EXTRACTOR_PROMPT = SystemMessage(
    "You are an AP specialist. Extract structured invoice data from the vendor invoice text. "
    "Extract vendor_id, invoice_number, invoice_date (YYYY-MM-DD), po_reference, "
    "total_amount, currency, and all line items with description, quantity, unit_price, line_total."
)

MATCHER_PROMPT = SystemMessage(
    "You are an AP controller performing a 3-way match.\n"
    "Discrepancy rules:\n"
    "- quantity_short (warn): GR qty < invoice qty\n"
    "- price_variance (block if >2%): invoice unit_price differs from PO unit_price by >2%\n"
    "- gr_missing (block): no GR record exists\n"
    "- po_not_found (block): no PO record exists\n"
    "Approval tiers:\n"
    "- auto_approve: clean + total < 10000\n"
    "- line_manager: warn + total < 25000\n"
    "- finance_controller: (warn + total >= 25000) or (block + total < 50000)\n"
    "- vp_finance: block + total >= 50000\n"
    "match_status: clean=no issues, discrepancy=warn only, blocked=any block-level issue."
)


def create_extractor():
    llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
    return EXTRACTOR_PROMPT | llm.with_structured_output(ExtractedInvoice)


def create_matcher():
    llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
    return MATCHER_PROMPT | llm.with_structured_output(MatchResult)


def run(invoice_text):
    invoice = create_extractor().invoke(invoice_text)
    po_data = lookup_po(invoice.po_reference)
    gr_data = lookup_gr(invoice.po_reference)
    context = (
        f"Invoice:\n{json.dumps(invoice.model_dump(), indent=2)}\n\n"
        f"Purchase Order:\n{json.dumps(po_data, indent=2) if po_data else 'NOT FOUND'}\n\n"
        f"Goods Receipt:\n{json.dumps(gr_data, indent=2) if gr_data else 'NOT FOUND'}"
    )
    match_result = create_matcher().invoke(context)
    return {"invoice": invoice, "po_data": po_data, "gr_data": gr_data, "match_result": match_result}


print("Workflow ready.")

## Part 5 — Run a Clean Invoice

ACME-001 invoices 4 cloud servers at $2,100 each = $8,400.
PO and GR both show 4 units at $2,100. Expected: **clean / auto_approve**.

In [ ]:
INVOICE_CLEAN = """
ACME Tech Supplies
Vendor ID: ACME-001
Invoice Number: INV-ACME-2025-0892
Invoice Date: 2025-06-15
PO Reference: PO-2025-001

Line Items:
  1. Cloud Server Model X500 -- Qty: 4 -- Unit Price: $2,100.00 -- Total: $8,400.00

Invoice Total: $8,400.00 USD
"""

result = run(INVOICE_CLEAN)
mr = result["match_result"]
inv = result["invoice"]

print(f"Vendor       : {inv.vendor_id}")
print(f"Invoice#     : {inv.invoice_number}")
print(f"Total        : {inv.currency} {inv.total_amount:,.2f}")
print(f"Match Status : {mr.match_status.upper()}")
print(f"Approval Tier: {mr.approval_tier.upper()}")
print(f"Rationale    : {mr.approval_rationale}")

## Part 6 — Run an Invoice with a Discrepancy

BETA-002 bills for 100 chairs at $150 each = $15,000.
But the goods receipt shows only 85 chairs were delivered.
Expected: **discrepancy / quantity_short (warn) / line_manager**.

In [ ]:
INVOICE_SHORT = """
Beta Office Solutions
Vendor ID: BETA-002
Invoice Number: INV-BETA-2025-0441
Invoice Date: 2025-06-18
PO Reference: PO-2025-002

Line Items:
  1. Ergonomic Office Chair -- Qty: 100 -- Unit Price: $150.00 -- Total: $15,000.00

Invoice Total: $15,000.00 USD
"""

result2 = run(INVOICE_SHORT)
mr2 = result2["match_result"]
inv2 = result2["invoice"]

print(f"Vendor       : {inv2.vendor_id}")
print(f"Total        : {inv2.currency} {inv2.total_amount:,.2f}")
print(f"Match Status : {mr2.match_status.upper()}")
print("Discrepancies:")
for d in mr2.discrepancies:
    print(f"  [{d.severity.upper()}] {d.discrepancy_type}: invoice={d.invoice_value} expected={d.expected_value}")
print(f"Approval Tier: {mr2.approval_tier.upper()}")
print(f"Rationale    : {mr2.approval_rationale}")

## Part 7 — Exercise: What Happens When the PO Does Not Exist?

What happens when an invoice references a PO that does not exist at all?

**Your turn:** The invoice below uses `PO-9999-999` which is not in the mock data.

Questions to answer before running:
1. Which `discrepancy_type` fires?
2. What `severity` is assigned?
3. What `approval_tier` results, and why?

Run the cell and check your predictions against the answer key.

In [ ]:
INVOICE_UNKNOWN_PO = """
Mystery Vendor Ltd
Vendor ID: MYSTERY-099
Invoice Number: INV-MYST-2025-0001
Invoice Date: 2025-06-25
PO Reference: PO-9999-999

Line Items:
  1. Unknown Goods -- Qty: 10 -- Unit Price: $500.00 -- Total: $5,000.00

Invoice Total: $5,000.00 USD
"""

result_ex = run(INVOICE_UNKNOWN_PO)
mr_ex = result_ex["match_result"]

print(f"Match Status : {mr_ex.match_status.upper()}")
print("Discrepancies:")
for d in mr_ex.discrepancies:
    print(f"  [{d.severity.upper()}] {d.discrepancy_type}")
print(f"Approval Tier: {mr_ex.approval_tier.upper()}")
print(f"Rationale    : {mr_ex.approval_rationale}")

# Answer key
print("\n--- Answer Key ---")
print("discrepancy_type : po_not_found")
print("severity         : block")
print("match_status     : blocked")
print("approval_tier    : finance_controller  (block + total $5,000 < $50,000)")